# Notebook 02 — Cointegration Scan (PROTOTYPE)

**Purpose:** Test ~45 equity pairs (from 10 prototype tickers) for cointegration
using the Engle-Granger framework, apply BH-FDR correction, compute half-life,
filter by economic logic, and produce a Pairs Selection Report.

**Scope:** Prototype — 10 tickers, C(10,2) = 45 pairs. The full-run version
will scale to ~50 tickers / ~1,225 pairs.

**Methodology source of truth:**
- `.agents/workflows/cointegration_methodology_spec.md`
- `.agents/workflows/implementation_checklist.md`

**Key workflow (already approved, not revisited here):**
- `coint()` = official cointegration verdict (p-value, test statistic)
- OLS = hedge ratio + spread construction + half-life + plots
- BH-FDR at q=0.05 for multiple testing correction
- Hard filters: BH-FDR pass, half-life [5,60] days, beta > 0, economic logic
- Fallback: relax half-life to [3,90] if < 10 survive, then FDR to q=0.10 if < 5

## 1. Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
import warnings
import sys, io

# Fix Windows console encoding
if sys.stdout.encoding != 'utf-8':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

warnings.filterwarnings('ignore', category=FutureWarning)

from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# -- Paths --
PROJECT_ROOT = Path(r"d:\Quant Finance\Quant Program\Week 1")
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "pair_scan_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- Approved methodology parameters --
COINT_MAXLAG = 30          # Cap ADF lag length inside coint()
COINT_TREND = 'c'          # Include constant in cointegrating regression
COINT_AUTOLAG = 'aic'      # Lag selection method
BH_FDR_ALPHA = 0.05        # Benjamini-Hochberg FDR level
HALF_LIFE_RANGE = (5, 60)  # Trading days — hard filter
HALF_LIFE_FALLBACK = (3, 90)  # Relaxed range if < 10 pairs survive
BARS_PER_DAY = 77          # 5-min bars per full trading day (from Notebook 01 diagnostic)
MIN_ALIGNED_OBS = 5000     # Skip pairs with fewer aligned observations

print(f"Project root: {PROJECT_ROOT}")
print(f"BH-FDR alpha: {BH_FDR_ALPHA}")
print(f"Half-life range: {HALF_LIFE_RANGE} trading days")
print(f"Bars per day: {BARS_PER_DAY}")

## 2. Data Loading and Quick Validation

In [ ]:
# Load the cleaned 5-min log-price panel from Notebook 01
panel = pd.read_parquet(INTERMEDIATE_DIR / "log_prices_5min.parquet")

# -- Validation checks (fail loudly if panel is not clean) --
assert isinstance(panel.index, pd.DatetimeIndex), "Index is not DatetimeIndex"
assert panel.index.is_unique, "Index has duplicates"
assert panel.index.is_monotonic_increasing, "Index is not sorted"
assert panel.isna().sum().sum() == 0, "Panel contains NaN"
assert all(panel.dtypes == np.float64), "Not all columns are float64"

# Handle timezone
if panel.index.tz is None:
    panel.index = panel.index.tz_localize('US/Eastern')
    print("NOTE: Re-localized index to US/Eastern (parquet stripped tz)")

tickers = list(panel.columns)
n_tickers = len(tickers)
n_rows = len(panel)

print(f"\nPanel loaded successfully:")
print(f"  Shape: {panel.shape}")
print(f"  Tickers: {tickers}")
print(f"  Date range: {panel.index[0].date()} to {panel.index[-1].date()}")
print(f"  Index tz: {panel.index.tz}")
print(f"  NaN: 0")
print()
panel.head(3)

## 3. Universe Metadata and Sector Mapping

Sector mapping defined in a single cell per the implementation checklist.
For the prototype, this covers only the 10 tickers.
For the full run, this dict must cover all ~317 possible tickers.

In [ ]:
# -- SECTOR MAPPING (single dedicated cell) --
# Prototype-only: 10 tickers. Full run must expand to all surviving tickers.
SECTOR_MAP = {
    'AAPL':  'Technology',
    'MSFT':  'Technology',
    'GOOGL': 'Technology',
    'AMZN':  'Consumer Discretionary',
    'XOM':   'Energy',
    'CVX':   'Energy',
    'JPM':   'Financials',
    'BAC':   'Financials',
    'V':     'Financials',
    'MA':    'Financials',
}

# -- Hard assertion: every ticker in the panel must have a sector mapping --
unmapped = set(tickers) - set(SECTOR_MAP.keys())
assert len(unmapped) == 0, f"Unmapped tickers: {unmapped}. Add them to SECTOR_MAP."
print(f"Sector mapping covers all {n_tickers} tickers.")

# -- Economic rationale lookup for known within-sector pairs --
# Used in Stage 4 (economic logic filter) to auto-assign rationale
ECON_RATIONALE = {
    'AAPL-MSFT':  ('Tier 1', 'Same sector (Technology), both mega-cap platform companies'),
    'AAPL-GOOGL': ('Tier 1', 'Same sector (Technology), both mega-cap with hardware/services overlap'),
    'MSFT-GOOGL': ('Tier 1', 'Same sector (Technology), direct competitors in cloud and enterprise'),
    'XOM-CVX':    ('Tier 1', 'Same sub-industry (Integrated Oil & Gas), identical crude exposure'),
    'JPM-BAC':    ('Tier 1', 'Same sub-industry (Diversified Banks), same Fed rate and credit cycle'),
    'V-MA':       ('Tier 1', 'Same sub-industry (Payment Networks), duopoly with shared merchant base'),
    'JPM-V':      ('Tier 2', 'Same sector (Financials), bank vs payment network, shared consumer credit'),
    'JPM-MA':     ('Tier 2', 'Same sector (Financials), bank vs payment network, shared consumer credit'),
    'BAC-V':      ('Tier 2', 'Same sector (Financials), bank vs payment network, shared consumer credit'),
    'BAC-MA':     ('Tier 2', 'Same sector (Financials), bank vs payment network, shared consumer credit'),
}

print(f"Economic rationale lookup: {len(ECON_RATIONALE)} pair rationales defined")

## 4. Pair Generation

Generate all C(10,2) = 45 unique pairs. Tag each with sector info.
Pair IDs use alphabetical ordering for consistency.

In [ ]:
pairs = []
for a, b in combinations(sorted(tickers), 2):
    pair_id = f"{a}-{b}"
    sector_a = SECTOR_MAP[a]
    sector_b = SECTOR_MAP[b]
    within_sector = sector_a == sector_b

    pairs.append({
        'pair_id': pair_id,
        'ticker_a': a,
        'ticker_b': b,
        'sector_a': sector_a,
        'sector_b': sector_b,
        'within_sector': within_sector,
    })

pairs_df = pd.DataFrame(pairs)
n_pairs = len(pairs_df)
n_within = pairs_df['within_sector'].sum()
n_cross = n_pairs - n_within

print(f"Pairs generated: {n_pairs} (expected C({n_tickers},2) = {n_tickers*(n_tickers-1)//2})")
print(f"  Within-sector: {n_within}")
print(f"  Cross-sector:  {n_cross}")
assert n_pairs == n_tickers * (n_tickers - 1) // 2, "Pair count mismatch"
assert pairs_df['pair_id'].is_unique, "Duplicate pair IDs"
print()
pairs_df.head(10)

## 5. Cointegration Testing Workflow

For each pair:
1. `coint()` -> cointegration test statistic + p-value (MacKinnon N=2)
2. `OLS` -> hedge ratio + spread construction
3. Store spread for later half-life computation

`coint()` is the official verdict. OLS is for spread diagnostics only.

In [ ]:
def compute_half_life(spread: pd.Series) -> float:
    """Compute OU half-life from spread series.

    Returns half-life in 5-min bars. Divide by BARS_PER_DAY for trading days.
    Returns np.nan if spread is not mean-reverting (lambda >= 0).
    """
    spread_lag = spread.shift(1)
    spread_diff = spread.diff()
    # Drop NaN from shift/diff
    valid = pd.concat([spread_diff, spread_lag], axis=1).dropna()
    valid.columns = ['diff', 'lag']

    if len(valid) < 100:
        return np.nan

    try:
        model = sm.OLS(valid['diff'], sm.add_constant(valid['lag'])).fit()
        lam = model.params['lag']
        if lam >= 0:
            return np.nan  # Not mean-reverting
        half_life_bars = -np.log(2) / lam
        return half_life_bars
    except Exception:
        return np.nan


def count_zero_crossings(spread: pd.Series) -> int:
    """Count how many times the demeaned spread crosses zero."""
    demeaned = spread - spread.mean()
    signs = np.sign(demeaned)
    # Remove zeros (treat as continuation of prior sign)
    signs = signs[signs != 0]
    crossings = (signs.diff().abs() == 2).sum()
    return int(crossings)


# -- Run the scan --
scan_results = []
spreads = {}  # Store spreads for plotting later

print(f"Scanning {n_pairs} pairs...")
for i, row in pairs_df.iterrows():
    pair_id = row['pair_id']
    ta, tb = row['ticker_a'], row['ticker_b']

    log_a = panel[ta]
    log_b = panel[tb]

    # Inner join (already aligned in Notebook 01, but be explicit)
    aligned = pd.concat([log_a, log_b], axis=1).dropna()
    n_obs = len(aligned)

    if n_obs < MIN_ALIGNED_OBS:
        scan_results.append({
            'pair_id': pair_id, 'ticker_a': ta, 'ticker_b': tb,
            'sector_a': row['sector_a'], 'sector_b': row['sector_b'],
            'within_sector': row['within_sector'],
            'n_aligned_obs': n_obs, 'coint_tstat': np.nan,
            'raw_pval': np.nan, 'hedge_ratio': np.nan, 'intercept': np.nan,
            'scan_status': 'skipped: insufficient data',
        })
        continue

    series_a = aligned.iloc[:, 0].values
    series_b = aligned.iloc[:, 1].values

    # -- Step 1: coint() for official verdict --
    try:
        coint_t, pvalue, crit_values = coint(
            series_a, series_b,
            trend=COINT_TREND, autolag=COINT_AUTOLAG, maxlag=COINT_MAXLAG
        )
    except Exception as e:
        scan_results.append({
            'pair_id': pair_id, 'ticker_a': ta, 'ticker_b': tb,
            'sector_a': row['sector_a'], 'sector_b': row['sector_b'],
            'within_sector': row['within_sector'],
            'n_aligned_obs': n_obs, 'coint_tstat': np.nan,
            'raw_pval': np.nan, 'hedge_ratio': np.nan, 'intercept': np.nan,
            'scan_status': f'coint() error: {e}',
        })
        continue

    # -- Step 2: OLS for hedge ratio and spread --
    try:
        ols_model = sm.OLS(series_a, sm.add_constant(series_b)).fit()
        intercept = ols_model.params[0]
        hedge_ratio = ols_model.params[1]
        spread = pd.Series(
            series_a - hedge_ratio * series_b,
            index=aligned.index
        )
        spreads[pair_id] = spread
    except Exception as e:
        scan_results.append({
            'pair_id': pair_id, 'ticker_a': ta, 'ticker_b': tb,
            'sector_a': row['sector_a'], 'sector_b': row['sector_b'],
            'within_sector': row['within_sector'],
            'n_aligned_obs': n_obs, 'coint_tstat': coint_t,
            'raw_pval': pvalue, 'hedge_ratio': np.nan, 'intercept': np.nan,
            'scan_status': f'OLS error: {e}',
        })
        continue

    scan_results.append({
        'pair_id': pair_id, 'ticker_a': ta, 'ticker_b': tb,
        'sector_a': row['sector_a'], 'sector_b': row['sector_b'],
        'within_sector': row['within_sector'],
        'n_aligned_obs': n_obs, 'coint_tstat': coint_t,
        'raw_pval': pvalue, 'hedge_ratio': hedge_ratio,
        'intercept': intercept, 'scan_status': 'OK',
    })

scan_df = pd.DataFrame(scan_results)
n_ok = (scan_df['scan_status'] == 'OK').sum()
n_failed = n_pairs - n_ok
print(f"\nScan complete: {n_ok} OK, {n_failed} failed/skipped")
print(f"Raw p-value range: {scan_df['raw_pval'].min():.6f} to {scan_df['raw_pval'].max():.4f}")

# Show top 10 by raw p-value
print("\nTop 10 pairs by raw p-value (most significant first):")
top10 = scan_df[scan_df['scan_status'] == 'OK'].nsmallest(10, 'raw_pval')
print(top10[['pair_id', 'within_sector', 'coint_tstat', 'raw_pval', 'hedge_ratio']].to_string(index=False))

## 6. Statistical Filtering

### 6a. BH-FDR Correction

In [ ]:
# Only include pairs with valid p-values in the correction
valid_mask = scan_df['scan_status'] == 'OK'
valid_pvals = scan_df.loc[valid_mask, 'raw_pval'].values
n_valid = len(valid_pvals)

print(f"Applying BH-FDR correction to {n_valid} valid p-values (q={BH_FDR_ALPHA})...")

reject, pvals_adj, _, _ = multipletests(valid_pvals, alpha=BH_FDR_ALPHA, method='fdr_bh')

# Write results back to scan_df
scan_df['bh_adj_pval'] = np.nan
scan_df['bh_reject'] = False
scan_df.loc[valid_mask, 'bh_adj_pval'] = pvals_adj
scan_df.loc[valid_mask, 'bh_reject'] = reject

n_bh_pass = reject.sum()
print(f"BH-FDR results: {n_bh_pass} pairs rejected null (cointegrated) out of {n_valid} tested")

# Correctness check: adjusted p-values >= raw p-values
if n_bh_pass > 0:
    adj_vs_raw = scan_df.loc[valid_mask, ['raw_pval', 'bh_adj_pval']].dropna()
    assert (adj_vs_raw['bh_adj_pval'] >= adj_vs_raw['raw_pval'] - 1e-10).all(), \
        "BH-adjusted p-values should be >= raw p-values"
    print("Correctness check: adjusted p-values >= raw p-values PASSED")

### 6b. Half-Life Computation (for BH-passing pairs only)

In [ ]:
# Compute half-life only for pairs that passed BH-FDR
scan_df['half_life_bars'] = np.nan
scan_df['half_life_days'] = np.nan
scan_df['zero_crossings'] = np.nan

bh_passing_ids = scan_df[scan_df['bh_reject'] == True]['pair_id'].tolist()

if len(bh_passing_ids) == 0:
    print("No pairs passed BH-FDR. Half-life computation skipped.")
else:
    print(f"Computing half-life for {len(bh_passing_ids)} BH-passing pairs...")
    for pair_id in bh_passing_ids:
        if pair_id not in spreads:
            continue
        spread = spreads[pair_id]

        hl_bars = compute_half_life(spread)
        hl_days = hl_bars / BARS_PER_DAY if not np.isnan(hl_bars) else np.nan
        zc = count_zero_crossings(spread)

        idx = scan_df[scan_df['pair_id'] == pair_id].index[0]
        scan_df.loc[idx, 'half_life_bars'] = hl_bars
        scan_df.loc[idx, 'half_life_days'] = hl_days
        scan_df.loc[idx, 'zero_crossings'] = zc

    # Show BH-passing pairs with their half-lives
    bh_passing = scan_df[scan_df['bh_reject'] == True].copy()
    print("\nBH-passing pairs with diagnostics:")
    display_cols = ['pair_id', 'within_sector', 'raw_pval', 'bh_adj_pval',
                    'hedge_ratio', 'half_life_days', 'zero_crossings']
    print(bh_passing[display_cols].to_string(index=False))

### 6c. Hard Filters (sequential funnel)

In [ ]:
# -- Build the filter funnel --
funnel = []

# Stage 0: All tested
all_tested = scan_df[scan_df['scan_status'] == 'OK'].copy()
n_stage0 = len(all_tested)
funnel.append(('All tested', n_stage0))

# Stage 1: BH-FDR
stage1 = all_tested[all_tested['bh_reject'] == True].copy()
n_stage1 = len(stage1)
funnel.append(('BH-FDR (q=0.05)', n_stage1))

# Stage 2: Half-life in [5, 60] trading days
hl_min, hl_max = HALF_LIFE_RANGE
stage2 = stage1[
    (stage1['half_life_days'] >= hl_min) &
    (stage1['half_life_days'] <= hl_max)
].copy()
n_stage2 = len(stage2)
funnel.append((f'Half-life [{hl_min},{hl_max}]d', n_stage2))

# Stage 3: Hedge ratio > 0
stage3 = stage2[stage2['hedge_ratio'] > 0].copy()
n_stage3 = len(stage3)
funnel.append(('Hedge ratio > 0', n_stage3))

# -- Fallback logic --
FALLBACK_USED = False
filter_regime = 'primary'

if n_stage3 < 10:
    print(f"\nFallback check: {n_stage3} pairs after Stage 3 (< 10 threshold)")
    hl_fb_min, hl_fb_max = HALF_LIFE_FALLBACK

    # If BH-FDR already let nothing through, relax FDR first
    if n_stage1 == 0 or (n_stage3 < 5):
        print(f"  Relaxing BH-FDR to q=0.10...")
        reject_fb, pvals_adj_fb, _, _ = multipletests(valid_pvals, alpha=0.10, method='fdr_bh')
        scan_df.loc[valid_mask, 'bh_adj_pval'] = pvals_adj_fb
        scan_df.loc[valid_mask, 'bh_reject'] = reject_fb
        FALLBACK_USED = True
        filter_regime = 'relaxed'
        n_bh_pass_fb = reject_fb.sum()
        print(f"  BH-FDR at q=0.10: {n_bh_pass_fb} pairs pass")

        # Compute half-life for newly passing pairs
        new_bh_ids = scan_df[(scan_df['bh_reject'] == True) & (scan_df['half_life_days'].isna())]['pair_id'].tolist()
        for pid in new_bh_ids:
            if pid in spreads:
                hl_bars = compute_half_life(spreads[pid])
                hl_days = hl_bars / BARS_PER_DAY if not np.isnan(hl_bars) else np.nan
                zc = count_zero_crossings(spreads[pid])
                idx = scan_df[scan_df['pair_id'] == pid].index[0]
                scan_df.loc[idx, 'half_life_bars'] = hl_bars
                scan_df.loc[idx, 'half_life_days'] = hl_days
                scan_df.loc[idx, 'zero_crossings'] = zc

        # Re-run stages 1-3 with relaxed thresholds
        stage1 = scan_df[(scan_df['scan_status'] == 'OK') & (scan_df['bh_reject'] == True)].copy()
        n_stage1 = len(stage1)
        stage2 = stage1[
            (stage1['half_life_days'] >= hl_fb_min) &
            (stage1['half_life_days'] <= hl_fb_max)
        ].copy()
        n_stage2 = len(stage2)
        stage3 = stage2[stage2['hedge_ratio'] > 0].copy()
        n_stage3 = len(stage3)
        print(f"  After relaxation: FDR={n_stage1}, HL=[{hl_fb_min},{hl_fb_max}]={n_stage2}, beta>0={n_stage3}")

        # Update funnel
        funnel = [
            ('All tested', n_stage0),
            (f'BH-FDR (q=0.10 RELAXED)', n_stage1),
            (f'Half-life [{hl_fb_min},{hl_fb_max}]d (RELAXED)', n_stage2),
            ('Hedge ratio > 0', n_stage3),
        ]
    else:
        # BH passed some pairs but fewer than 10 survived HL+HR — relax HL only
        stage2_fb = stage1[
            (stage1['half_life_days'] >= hl_fb_min) &
            (stage1['half_life_days'] <= hl_fb_max)
        ].copy()
        stage3_fb = stage2_fb[stage2_fb['hedge_ratio'] > 0].copy()

        if len(stage3_fb) > n_stage3:
            print(f"  Relaxing half-life to [{hl_fb_min},{hl_fb_max}]: {len(stage3_fb)} pairs (was {n_stage3})")
            stage3 = stage3_fb
            n_stage3 = len(stage3)
            FALLBACK_USED = True
            filter_regime = 'relaxed'
            funnel[-2] = (f'Half-life [{hl_fb_min},{hl_fb_max}]d (RELAXED)', len(stage2_fb))
            funnel[-1] = ('Hedge ratio > 0 (after relaxation)', n_stage3)

# Print funnel
funnel.append(('Pre-economic logic', n_stage3))
print("\n-- Filter Funnel --")
for stage_name, count in funnel:
    print(f"  {stage_name}: {count}")

# Print filter regime
if not FALLBACK_USED:
    print("\nMAIN RESULT: All filters applied at primary thresholds (BH q=0.05, HL=[5,60]).")
else:
    print(f"\nSENSITIVITY RELAXATION APPLIED: Thresholds were relaxed because fewer than 10 pairs survived primary filters.")
    print("Results below reflect relaxed thresholds and should be interpreted with additional caution.")

## 7. Economic Logic Filter

For each pair surviving Stage 3: assign economic rationale or reject.
Within-sector pairs get auto-assigned rationale from the lookup.
Cross-sector pairs are rejected unless they have a documented economic link.

In [ ]:
econ_results = []

if len(stage3) == 0:
    print("No pairs reached the economic logic stage.")
    n_econ_pass = 0
    n_econ_reject = 0
else:
    for _, row in stage3.iterrows():
        pair_id = row['pair_id']
        within = row['within_sector']

        if pair_id in ECON_RATIONALE:
            tier, rationale = ECON_RATIONALE[pair_id]
            econ_results.append({
                'pair_id': pair_id,
                'economic_tier': tier,
                'economic_rationale': rationale,
                'economic_pass': True,
            })
        elif within:
            # Within-sector but not in the explicit lookup -- assign generic rationale
            sector = row['sector_a']
            econ_results.append({
                'pair_id': pair_id,
                'economic_tier': 'Tier 2',
                'economic_rationale': f'Same sector ({sector}), shared macro exposure',
                'economic_pass': True,
            })
        else:
            # Cross-sector, no documented link
            econ_results.append({
                'pair_id': pair_id,
                'economic_tier': 'Reject',
                'economic_rationale': 'Cross-sector, no identifiable economic linkage',
                'economic_pass': False,
            })

    n_econ_pass = sum(r['economic_pass'] for r in econ_results)
    n_econ_reject = len(econ_results) - n_econ_pass
    print(f"Economic logic filter: {n_econ_pass} approved, {n_econ_reject} rejected")

econ_df = pd.DataFrame(econ_results) if econ_results else pd.DataFrame(
    columns=['pair_id', 'economic_tier', 'economic_rationale', 'economic_pass']
)
funnel.append(('Economic logic', n_econ_pass))

if len(econ_df) > 0:
    print(econ_df.to_string(index=False))

## 8. Output Tables

In [ ]:
# -- Table 2: Full Scan Results --
full_scan = scan_df.copy()
full_scan_path = OUTPUT_DIR / "full_pair_scan_results.parquet"
full_scan.to_parquet(full_scan_path, index=False)
print(f"Saved: {full_scan_path} ({len(full_scan)} rows)")

# -- Table 3: Approved Pairs --
approved_ids = econ_df[econ_df['economic_pass'] == True]['pair_id'].tolist()
approved = stage3[stage3['pair_id'].isin(approved_ids)].copy()
approved = approved.merge(econ_df, on='pair_id', how='left')
approved['filter_regime'] = filter_regime

# Rank by BH-adjusted p-value (simplified ranking per methodology)
approved = approved.sort_values('bh_adj_pval').reset_index(drop=True)
approved['rank'] = range(1, len(approved) + 1)

approved_path = OUTPUT_DIR / "approved_pairs.parquet"
approved.to_parquet(approved_path, index=False)
print(f"Saved: {approved_path} ({len(approved)} rows)")

if len(approved) > 0:
    print("\n-- Approved Pairs (Table 3) --")
    display_cols = ['rank', 'pair_id', 'within_sector', 'coint_tstat', 'raw_pval',
                    'bh_adj_pval', 'hedge_ratio', 'half_life_days', 'zero_crossings',
                    'economic_tier', 'economic_rationale', 'filter_regime']
    print(approved[display_cols].to_string(index=False))
else:
    print("\nNo pairs survived all filters. This is a valid result.")

# -- Table 4: Rejected Pairs Summary --
# Build rejection funnel table
rejection_data = []
stages = [
    ('BH-FDR', n_stage0, n_stage0 - n_stage1),
    ('Half-life', n_stage1, n_stage1 - n_stage2 if not FALLBACK_USED else n_stage1 - len(stage2)),
    ('Hedge ratio', n_stage2 if not FALLBACK_USED else len(stage2), n_stage2 - n_stage3 if not FALLBACK_USED else len(stage2) - n_stage3),
    ('Economic logic', n_stage3, n_econ_reject),
]

running = n_stage0
for stage_name, entering, rejected in stages:
    remaining = entering - rejected
    # Find an example rejected pair for this stage
    example_pair = ''
    example_reason = ''
    if stage_name == 'BH-FDR':
        non_passing = all_tested[all_tested['bh_reject'] == False]
        if len(non_passing) > 0:
            example_pair = non_passing.iloc[0]['pair_id']
            example_reason = f"bh_adj_pval={non_passing.iloc[0]['bh_adj_pval']:.4f} > {BH_FDR_ALPHA}"
    elif stage_name == 'Economic logic':
        econ_rejects = econ_df[econ_df['economic_pass'] == False]
        if len(econ_rejects) > 0:
            example_pair = econ_rejects.iloc[0]['pair_id']
            example_reason = econ_rejects.iloc[0]['economic_rationale']

    rejection_data.append({
        'filter_stage': stage_name,
        'pairs_entering': entering,
        'pairs_rejected': rejected,
        'pairs_remaining': remaining,
        'example_rejected_pair': example_pair,
        'example_reason': example_reason,
    })

rejection_df = pd.DataFrame(rejection_data)
rejection_path = OUTPUT_DIR / "rejected_pairs_summary.parquet"
rejection_df.to_parquet(rejection_path, index=False)
print(f"\nSaved: {rejection_path}")
print("\n-- Rejection Summary (Table 4) --")
print(rejection_df.to_string(index=False))

## 9. Visual Validation

Plot representative pairs to audit whether the workflow is behaving sensibly.

In [ ]:
import matplotlib.pyplot as plt

def plot_pair(pair_id: str, scan_row: pd.Series, spread: pd.Series, label: str):
    """Plot log prices and spread for a single pair."""
    ta, tb = scan_row['ticker_a'], scan_row['ticker_b']
    hr = scan_row['hedge_ratio']
    pval = scan_row['raw_pval']
    hl = scan_row.get('half_life_days', np.nan)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Top: normalized log prices
    log_a = panel[ta]
    log_b = panel[tb]
    # Align to spread's index
    common_idx = spread.index
    norm_a = log_a.loc[common_idx] - log_a.loc[common_idx].iloc[0]
    norm_b = log_b.loc[common_idx] - log_b.loc[common_idx].iloc[0]

    axes[0].plot(norm_a.index, norm_a.values, label=ta, alpha=0.8)
    axes[0].plot(norm_b.index, norm_b.values, label=tb, alpha=0.8)
    axes[0].set_ylabel('Normalized log price')
    axes[0].legend()
    hl_str = f'{hl:.1f}d' if not np.isnan(hl) else 'N/A'
    axes[0].set_title(f'[{label}] {pair_id} | p={pval:.4f} | beta={hr:.3f} | HL={hl_str}')
    axes[0].grid(True, alpha=0.3)

    # Bottom: spread with sigma bands
    s_mean = spread.mean()
    s_std = spread.std()
    axes[1].plot(spread.index, spread.values, color='steelblue', alpha=0.7, linewidth=0.5)
    axes[1].axhline(s_mean, color='black', linestyle='-', linewidth=1, label='Mean')
    axes[1].axhline(s_mean + s_std, color='red', linestyle='--', alpha=0.6, label='+/- 1 sigma')
    axes[1].axhline(s_mean - s_std, color='red', linestyle='--', alpha=0.6)
    axes[1].axhline(s_mean + 2*s_std, color='darkred', linestyle=':', alpha=0.4, label='+/- 2 sigma')
    axes[1].axhline(s_mean - 2*s_std, color='darkred', linestyle=':', alpha=0.4)
    axes[1].set_ylabel(f'Spread ({ta} - {hr:.2f}*{tb})')
    axes[1].set_xlabel('Date')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# -- Plot approved pairs (strongest first) --
if len(approved) > 0:
    print("Plotting approved pairs (top by p-value):")
    for _, arow in approved.head(5).iterrows():
        pid = arow['pair_id']
        if pid in spreads:
            srow = scan_df[scan_df['pair_id'] == pid].iloc[0]
            plot_pair(pid, srow, spreads[pid], 'APPROVED')

# -- Plot at least one rejected pair for comparison --
rejected_pids = scan_df[
    (scan_df['scan_status'] == 'OK') &
    (scan_df['bh_reject'] == False)
].nlargest(1, 'raw_pval')

if len(rejected_pids) > 0:
    print("\nPlotting a rejected pair (highest p-value) for comparison:")
    for _, rrow in rejected_pids.iterrows():
        pid = rrow['pair_id']
        if pid in spreads:
            plot_pair(pid, rrow, spreads[pid], 'REJECTED')

### P-value Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
valid_pvals_plot = scan_df[scan_df['scan_status'] == 'OK']['raw_pval'].dropna()
ax.hist(valid_pvals_plot, bins=30, edgecolor='white', alpha=0.7)
ax.axvline(0.05, color='red', linestyle='--', label=f'alpha=0.05')
n_raw_sig = (valid_pvals_plot < 0.05).sum()
n_bh_sig = scan_df['bh_reject'].sum()
ax.set_xlabel('Raw p-value')
ax.set_ylabel('Count')
ax.set_title(f'P-value Distribution | {n_raw_sig} raw < 0.05 | {n_bh_sig} survive BH-FDR')
ax.legend()
plt.tight_layout()
plt.show()

### Rejection Funnel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
stage_names = [r['filter_stage'] for r in rejection_data]
stage_remaining = [r['pairs_remaining'] for r in rejection_data]
# Prepend the total
stage_names = ['All tested'] + stage_names
stage_remaining = [n_stage0] + stage_remaining

bars = ax.bar(range(len(stage_names)), stage_remaining, color='steelblue', edgecolor='white')
ax.set_xticks(range(len(stage_names)))
ax.set_xticklabels(stage_names, rotation=30, ha='right')
ax.set_ylabel('Pairs remaining')
ax.set_title('Filter Funnel')
for bar, val in zip(bars, stage_remaining):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Prototype Summary and Validation

In [ ]:
print("=" * 70)
print("NOTEBOOK 02 PROTOTYPE -- STATUS SUMMARY")
print("=" * 70)
print()
print(f"Input panel:             {panel.shape[0]} rows x {panel.shape[1]} tickers")
print(f"Date range:              {panel.index[0].date()} to {panel.index[-1].date()}")
print(f"Pairs generated:         {n_pairs}")
print(f"Pairs tested (OK):       {n_ok}")
print(f"Pairs failed/skipped:    {n_failed}")
print()
print("-- Filter Funnel --")
for stage_name, count in funnel:
    print(f"  {stage_name}: {count}")
print()
print(f"Filter regime:           {filter_regime}")
if FALLBACK_USED:
    print("  (Sensitivity relaxation was applied)")
print()
print(f"Approved pairs:          {len(approved)}")
if len(approved) > 0:
    print(f"  Best pair:             {approved.iloc[0]['pair_id']} (p={approved.iloc[0]['bh_adj_pval']:.6f})")
print()
print("Output files:")
print(f"  {full_scan_path}")
print(f"  {approved_path}")
print(f"  {rejection_path}")
print()
if len(approved) > 0:
    print("PROTOTYPE PIPELINE VALIDATED. Ready to scale to full run.")
else:
    print("No pairs survived. Pipeline logic is correct but prototype universe")
    print("may be too small or 2022 market conditions may not produce cointegration")
    print("for these specific tickers. This is a VALID result.")

### Assumptions and Notes

1. **Prototype only:** 10 tickers, 45 pairs. Full run scales to ~50 tickers, ~1,225 pairs.

2. **`coint()` is the official verdict.** OLS is used separately for hedge ratio,
   spread, half-life, and plots. These two roles are never mixed.

3. **BH-FDR at q=0.05** is the default. Fallback to q=0.10 only if fewer than
   5 pairs survive after half-life relaxation.

4. **Half-life conversion:** 1 trading day = 77 five-minute bars (from Notebook 01
   diagnostic: full session days have 77 bars).

5. **Economic logic** uses a hardcoded rationale lookup for the 10 prototype tickers.
   The full run will need a more comprehensive lookup covering all ~50 tickers.

6. **Hurst exponent:** Skipped in prototype per methodology (optional, not a gate).

7. **No backtesting, trading rules, or portfolio simulation in this notebook.**